# **Mount Drive**

# **Load Data**

In [3]:
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
from matplotlib import pyplot as plt
import seaborn as sns
from glob import glob

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)



In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import detrend

fpath_yield = '<DATA_ROOT>/Yield/crops_us_yield_agg_irig.csv'
dfy = pd.read_csv(fpath_yield)
df_grouped = dfy.groupby(['state_name','county_name','prodn_practice_desc','commodity_desc'])

def detrend_yield(yield_series):
    detrended_yield = detrend(yield_series)
    return pd.Series(detrended_yield, index=yield_series.index)

dfy['detrended_yield'] = df_grouped['value_yield'].transform(detrend_yield)

fpath_fips = '<DATA_ROOT>/Yield/yield_fips_code.csv'
dffipsy = pd.read_csv(fpath_fips).rename(columns={' county_code':'county_code'})
df_new = dffipsy.copy()
formatted_numbers = [str(n).zfill(3) for n in df_new['county_code'].values]
df_new['county_code'] = formatted_numbers
formatted_numbers = [str(n).zfill(2) for n in df_new['state_fips_code'].values]
df_new['state_fips_code'] = formatted_numbers
df_new['FIPS'] = df_new['state_fips_code'] + df_new['county_code']
dffipsy = df_new
dffipsy = dffipsy.drop(columns=['state_fips_code' , 'county_code'])
dffipsy

dfy = pd.merge(dfy,dffipsy,on=['state_name', 'county_name'])
dfy.iloc[0:5]

In [4]:
ffail = '<DATA_ROOT>/crop_failure_data/crops_failure_0595.csv'
df_fail = pd.read_csv(ffail)
df_fail = df_fail.drop(columns=['Unnamed: 0'])
df_fail = df_fail.sort_values(by=['FIPS'])
# cond1 = df_fail['year']>2011
cond2 = df_fail['Planted Acres'] > 0
df_fail = df_fail[cond2]
df_fail['fail_share'] = df_fail['Failed Acres'] /  df_fail['Planted Acres']
df_fail = df_fail[df_fail.fail_share < 1]
# df_fail = df_fail[df_fail.fail_share > 0]

formatted_numbers = [str(n).zfill(5) for n in df_fail['FIPS'].values]
df_fail['FIPS'] = formatted_numbers

# df_fail.year.unique()

In [ ]:
fsvi = '<DATA_ROOT>/SVI/SVI_timeseries.csv'
df_svi = pd.read_csv(fsvi)
df_svi = df_svi.sort_values(by=['FIPS'])
formatted_numbers = [str(n).zfill(5) for n in df_svi['FIPS'].values]
df_svi['FIPS'] = formatted_numbers
df_svi.iloc[0:5,:]

In [ ]:
fpath1 = '<DATA_ROOT>/WeatherIndex/Temp_Tercile_11nov.csv'
dft = pd.read_csv(fpath1)
dft = dft.rename(columns={'Year':'year'})
print(dft.iloc[0:2,:])

fpath2 = '<DATA_ROOT>/WeatherIndex/Drought_Map.csv'
dfd = pd.read_csv(fpath2)
dfd = dfd.rename(columns={'Year':'year', 'STATE':'State', 'COUNTY':'County'})
fpath = '<DATA_ROOT>/WeatherIndex/svi2020fips.csv'
df_fips_obj = pd.read_csv(fpath)[['FIPS',	'OBJECTID']]
dfd = pd.merge(dfd,df_fips_obj,on=['OBJECTID'])
print(dfd.iloc[0:2,:])

In [ ]:
drought_indices = ['PDSI', 'SPEI3', 'SPEI6', 'SPEI9', 'SPEI12', 'SPI3', 'SPI6', 'SPI9', 'SPI12']
for idx in drought_indices:
  dfd.loc[dfd[idx] == 0,idx] = 1
print(dfd.iloc[0:2,:])

HW_indices = ['Thr26', 'Thr27', 'Thr28', 'Thr29', 'Thr30', 'Thr31', 'Thr32', 'Thr33', 'Thr34', 'Thr35', 'Thr36']
for idx in HW_indices:
  dft.loc[dft[idx] == 0,idx] = 1
print(dft.iloc[0:2,:])


In [ ]:
shp_counties = '<DATA_ROOT>/SVI/Counties/'
counties = gpd.read_file(shp_counties)
# formatted_numbers = [int(n) for n in counties['FIPS'].values]
# counties['FIPS'] = formatted_numbers
counties.iloc[0:5,:]

,State,County,FIPS,geometry
0,Alabama,Autauga,01001,"POLYGON ((-86.92120 32.65754, -86.92035 32.658..."
1,Alabama,Baldwin,01003,"POLYGON ((-88.02858 30.22676, -88.02399 30.230..."
2,Alabama,Barbour,01005,"POLYGON ((-85.74803 31.61918, -85.74543 31.618..."
3,Alabama,Bibb,01007,"POLYGON ((-87.42194 33.00338, -87.31854 33.006..."
4,Alabama,Blount,01009,"POLYGON ((-86.96336 33.85822, -86.95967 33.857..."


In [ ]:
ClimReg_dir = '<DATA_ROOT>/US_ClimateRegions/US_ClimateRegions/'
gdf_clim = gpd.read_file(ClimReg_dir)
gdf_clim

In [ ]:
# import zipfile
# zip_path = '<DATA_ROOT>/CONUS_States/CONUS_STATES.zip'
# extract_dir = '<DATA_ROOT>/CONUS_States/'
# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_dir)

In [ ]:
states_dir = '<DATA_ROOT>/CONUS_States/'
gdf_states = gpd.read_file(states_dir)
gdf_states

# **Plot Treemap Failure Number - SVI - Irigation - Climate Region**

In [ ]:
bins = [0, 0.25, 0.5, 0.75, 1]
labels = [1, 2, 3, 4]
df_svi = df_svi.drop(columns=['State','County']).set_index(['FIPS','year'])
df_svi_cat = df_svi.apply(lambda x: pd.cut(x, bins=bins, labels=labels), axis=0)
df_svi_cat = df_svi_cat.reset_index()


In [ ]:
df_svi_cat = df_svi_cat[['FIPS','year','RPL_THEMES']]
cond_ig = df_fail['Irrigation Practice'] != 'ALL'
cond_fail =  df_fail.fail_share > 0
cols = ['FIPS', 'Crop', 'Irrigation Practice', 'year', 'fail_share']
df_merge = pd.merge(df_fail[cond_ig & cond_fail][cols],df_svi_cat,on=['FIPS','year'])

In [ ]:
fpath_counties_clim = '<DATA_ROOT>/US_ClimateRegions/Counties_ClimReg.csv'
df_counties_clim = pd.read_csv(fpath_counties_clim)
df_counties_clim = df_counties_clim.drop(columns=['Unnamed: 0',	'State', 	'County'])
formatted_numbers = [str(n).zfill(5) for n in df_counties_clim['FIPS'].values]
df_counties_clim['FIPS'] = formatted_numbers
df_counties_clim

In [ ]:
df_merge_clim = pd.merge(df_merge,df_counties_clim,on=['FIPS'])
grp_cols = ['ClimRegNam','Crop','Irrigation Practice','RPL_THEMES']
df_group = df_merge_clim.groupby(grp_cols).count().reset_index()
df_group = df_group[['ClimRegNam', 'Crop', 'Irrigation Practice', 'RPL_THEMES', 'FIPS']]
df_group['ir_svi'] = df_group['Irrigation Practice'] + df_group['RPL_THEMES'].astype(str)
df_group = df_group.rename(columns={'FIPS':'fail_count'})
df_group

In [ ]:
df_group.to_csv('/content/df_group.csv')

In [ ]:
import plotly.express as px

reds = ['#fff6ec','#fbd49b','#ed654d','#820f0a']
blues = ['#eff3ff','#bdd7e7','#6baed6','#2171b5']
ir_svi_colors = {
    'I1': blues[0], 'I2':blues[1], 'I3':blues[2], 'I4':blues[3],
    'N1': reds[0], 'N2': reds[1], 'N3': reds[2], 'N4': reds[3]
}

df = df_group[(df_group.Crop == 'CORN')]

fig = px.treemap(df,
                 path=[ 'ClimRegNam', 'ir_svi'],#px.Constant("US"),'ClimRegNam','Irrigation Practice', 'RPL_THEMES'
                 values='fail_count',
                 color='ir_svi',
                 color_discrete_map=ir_svi_colors
                 )
# colors_crop = {
#     'WHEAT': '#add8e6',       # Light Blue for young wheat, turning to '#f5deb3' for mature wheat
#     'CORN': '#008000',        # Dark Green for corn
#     'SOYBEANS': '#00ff00',    # Bright Green for soybeans
#     'COTTON': '#bdbd37',      # Light Yellow for cotton
#     'SORGHUM': '#ff4500',     # Reddish-Orange for sorghum
#     'HAY': '#785705',         # Dark Goldenrod for hay
#     'OATS': '#ccc9c4',        # Bisque for oats
#     'BARLEY': '#f56c1d'       # Sandy Brown for barley
# }

clim_colors = ['#fcfbfd','#efedf5','#dadaeb','#bcbddc','#9e9ac8','#807dba','#6a51a3','#54278f','#3f007d']
clim_reg_colors = dict(zip(list(df_group.ClimRegNam.unique()),clim_colors))
fig.for_each_trace(
    lambda t: t.update(
        marker_colors=[
            clim_reg_colors[id.split("/")[0]] if len(id.split("/")) == 1 else c
            for c, id in zip(t.marker.colors, t.ids)
        ]
    )
)

fig.update_layout(
    width=1000,  # Set the width of the figure in pixels
    height=1200,  # Set the height of the figure in pixels
)

fig.show()

In [ ]:

clim_reg_colors

In [ ]:
import plotly.express as px

df = px.data.tips()
fig = px.treemap(df, path=[px.Constant("all"), 'day', 'time', 'sex'], values='total_bill')
fig.update_traces(root_color="lightgrey")
fig.update_layout(margin=dict(t=50, l=25, r=25, b=25))
fig.update_traces(marker_line_width=0)

fig.show()


In [ ]:
import pandas as pd
import plotly.express as px
import random

# Sample DataFrame
data = pd.DataFrame({
    'clim': ['South']*32+['North']*32,
    'crop': ['Corn']*8+['Hay']*8+['Soybeans']*8+['Oats']*8+['Corn']*8+['Hay']*8+['Soybeans']*8+['Oats']*8,
    'irig': ['Ir']*4+['Rf']*4+['Ir']*4+['Rf']*4+['Ir']*4+['Rf']*4+['Ir']*4+['Rf']*4+['Ir']*4+['Rf']*4+['Ir']*4+['Rf']*4+['Ir']*4+['Rf']*4+['Ir']*4+['Rf']*4,
    'svi': [1,2,3,4]*16,
    'failure':[random.randint(1, 70) for _ in range(32)]+[random.randint(20, 130) for _ in range(32)]
})

reds = ['#fff6ec','#fbd49b','#ed654d','#820f0a']
blues = ['#eff3ff','#bdd7e7','#6baed6','#2171b5']
data['ir_svi'] = data['irig'] + data['svi'].astype(str)

custom_colors = {
    'Ir1': reds[0], 'Ir2':reds[1], 'Ir3':reds[2], 'Ir4':reds[3],
    'Rf1': blues[0], 'Rf2': blues[1], 'Rf3': blues[2], 'Rf4': blues[3]
}

fig = px.treemap(data, path=[px.Constant("US"),'clim', 'crop', 'irig', 'svi'],
                 values='failure',
                 color='ir_svi',
                 color_discrete_map=custom_colors
                 )


# colors_hex = {
#     'WHEAT': '#add8e6',       # Light Blue for young wheat, turning to '#f5deb3' for mature wheat
#     'CORN': '#008000',        # Dark Green for corn
#     'SOYBEANS': '#00ff00',    # Bright Green for soybeans
#     'COTTON': '#bdbd37',      # Light Yellow for cotton
#     'SORGHUM': '#ff4500',     # Reddish-Orange for sorghum
#     'HAY': '#785705',         # Dark Goldenrod for hay
#     'OATS': '#ccc9c4',        # Bisque for oats
#     'BARLEY': '#f56c1d'       # Sandy Brown for barley
# }

# Defining color map for countries

Crop_colors = {
    "Corn": "#008000",
    "Hay": "#785705",
    "Soybeans": "#00ff00",
    "Oats": "#ccc9c4"
}

Irig_colors = {
    "Ir": "#b3d9ff",
    "Rf": "#999900"
}

# ['#fcfbfd','#efedf5','#dadaeb','#bcbddc','#9e9ac8','#807dba','#6a51a3','#54278f','#3f007d']
clim_reg_colors = {
    "South": "#cccccc",
    "North": "#cccccc"
}
US_reg_colors = {
    "US": "#f0f0f0",
}

# Updating marker colors for countries
fig.for_each_trace(
    lambda t: t.update(
        marker_colors=[
            Crop_colors[id.split("/")[2]] if len(id.split("/")) == 3 else c
            for c, id in zip(t.marker.colors, t.ids)
        ]
    )
)

fig.for_each_trace(
    lambda t: t.update(
        marker_colors=[
            Irig_colors[id.split("/")[3]] if len(id.split("/")) == 4 else c
            for c, id in zip(t.marker.colors, t.ids)
        ]
    )
)

fig.for_each_trace(
    lambda t: t.update(
        marker_colors=[
            clim_reg_colors[id.split("/")[1]] if len(id.split("/")) == 2 else c
            for c, id in zip(t.marker.colors, t.ids)
        ]
    )
)

fig.for_each_trace(
    lambda t: t.update(
        marker_colors=[
            US_reg_colors[id.split("/")[0]] if len(id.split("/")) == 1 else c
            for c, id in zip(t.marker.colors, t.ids)
        ]
    )
)





fig.show()

# **Sankey Diagram**

In [ ]:
bins = [0, 0.25, 0.5, 0.75, 1]
labels = [1, 2, 3, 4]
df_svi = df_svi.drop(columns=['State','County']).set_index(['FIPS','year'])
df_svi_cat = df_svi.apply(lambda x: pd.cut(x, bins=bins, labels=labels), axis=0)
df_svi_cat = df_svi_cat.reset_index()


In [ ]:
df_svi_cat = df_svi_cat[['FIPS','year','RPL_THEMES']]
cond_ig = df_fail['Irrigation Practice'] != 'ALL'
cond_fail =  df_fail.fail_share > 0
cols = ['FIPS', 'Crop', 'Irrigation Practice', 'year', 'fail_share']
df_merge = pd.merge(df_fail[cond_ig & cond_fail][cols],df_svi_cat,on=['FIPS','year'])

In [ ]:
fpath_counties_clim = '<DATA_ROOT>/US_ClimateRegions/Counties_ClimReg.csv'
df_counties_clim = pd.read_csv(fpath_counties_clim)
df_counties_clim = df_counties_clim.drop(columns=['Unnamed: 0',	'State', 	'County'])
formatted_numbers = [str(n).zfill(5) for n in df_counties_clim['FIPS'].values]
df_counties_clim['FIPS'] = formatted_numbers
df_counties_clim

In [ ]:
df_merge_clim = pd.merge(df_merge,df_counties_clim,on=['FIPS'])
grp_cols = ['ClimRegNam','Crop','Irrigation Practice','RPL_THEMES']
df_group = df_merge_clim.groupby(grp_cols).count().reset_index()
df_group = df_group[['ClimRegNam', 'Crop', 'Irrigation Practice', 'RPL_THEMES', 'FIPS']]
# df_group['ir_svi'] = df_group['Irrigation Practice'] + df_group['RPL_THEMES'].astype(str)
df_group = df_group.rename(columns={'FIPS':'fail_count'})
df_group

In [ ]:
# df_group['RPL_THEMES'] = 5 - df_group['RPL_THEMES'].astype(int)
df_group['RPL_THEMES']  = df_group['RPL_THEMES'].map({4: 'Low',3: 'Low-Medium', 2: 'Medium-High', 1: 'High'})

df = df_group.groupby(['Crop', 'Irrigation Practice', 'RPL_THEMES']).sum().reset_index()
df1 = df.groupby(['Irrigation Practice' , 'Crop'])['fail_count'].sum().reset_index()
df1.columns = ['target', 'source', 'value']

df2 = df.groupby(['RPL_THEMES','Irrigation Practice' ])['fail_count'].sum().reset_index()
df2.columns = ['target', 'source', 'value']

df1['target'] = df1.target.map({'I': 'Irrigation', 'N': 'Rainfed'})
df2['source'] = df2.source.map({'I': 'Irrigation', 'N': 'Rainfed'})

# df2['target'] = df2.target.map({1: 'Low',2: 'Low-Medium', 3: 'Medium-High',4: 'High', })
links = pd.concat([df1, df2[::-1]], axis=0)#.reset_index().drop(columns=['index'])
# links.iloc[16:] = links.iloc[16:][::-1]
unique_source_target = list(pd.unique(links[['source', 'target']].values.ravel('K')))
# unique_source_target[-4:] = unique_source_target[-4:][::-1]
mapping_dict = {k: v for v, k in enumerate(unique_source_target)}
links['source'] = links['source'].map(mapping_dict)
links['target'] = links['target'].map(mapping_dict)
# links.iloc[16:] = links.iloc[16:][::-1]


colors_crop = {
    'WHEAT': '#add8e6',       # Light Blue for young wheat, turning to '#f5deb3' for mature wheat
    'CORN': '#008000',        # Dark Green for corn
    'SOYBEANS': '#00ff00',    # Bright Green for soybeans
    'COTTON': '#bdbd37',      # Light Yellow for cotton
    'SORGHUM': '#ff4500',     # Reddish-Orange for sorghum
    'HAY': '#785705',         # Dark Goldenrod for hay
    'OATS': '#ccc9c4',        # Bisque for oats
    'BARLEY': '#f56c1d'       # Sandy Brown for barley
}

sorted_dict_keys = {k: colors_crop[k] for k in sorted(colors_crop)}

links_dict = links.to_dict(orient='list')
import plotly.graph_objects as go
import pandas as pd
nodes1 = list(sorted_dict_keys.values())
nodes2 = ['#8080ff','#ff6666'][::-1]
nodes3 = ['#fff1e0','#fbd49b','#ed654d','#820f0a']

def hex_to_rgb(hex_color , i):
    hex_color = hex_color.lstrip('#')
    rgb = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    rgba = rgb + (0.85 - i/10,)
    return 'rgba' + str(rgba)

edges1 = ['rgba(128, 128, 255, 0.35)'] * 8
edges2 = ['rgba(255, 102, 102, 0.35)'] * 8
edges3 = ['#fff1e0','#fff1e0','#fbd49b','#fbd49b','#ed654d','#ed654d','#820f0a','#820f0a']
edges3 = [hex_to_rgb(c , i//2) for i , c in enumerate(edges3)]

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=unique_source_target,
        color = nodes1 + nodes2 + nodes3,
    ),
    link=dict(
        source=links_dict["source"],
        target=links_dict["target"],
        value=links_dict["value"],
        color = edges1 + edges2 + edges3
    )
)])

fig.update_layout(
    title_text="Failure Share",
    # font_family="Courier New",
    font_color="black",
    font_size=12,
    # title_font_family="Times New Roman",
    # title_font_color="blue",
    title_font_size=18,
)

fig.update_layout(
    width=1200,  # Set the width of the figure in pixels
    height=800,  # Set the height of the figure in pixels
)

fig.show()

import plotly.io as pio
import plotly.graph_objects as go


# out_path = '<DATA_ROOT>/Final_Exports/20231123-Section4/'
# # plt.savefig(out_path + 'Parcats_Diagramm_Crop_Irig_SVI' + '.png' , dpi=1200)
# pio.write_image(fig, out_path + 'Sankey_Diagramm_Crop_Irig_SVI' + '.png' , width=1200, height=800, scale=3)

# New Section

In [ ]:
unique_source_target

In [ ]:
import plotly.graph_objects as go
unique_list = ['home0', 'page_a0', 'page_b0', 'page_a1', 'page_b1',
               'page_c1', 'page_b2', 'page_a2', 'page_c2', 'page_c3']
sources = [0, 0, 1, 2, 2, 3, 3, 4, 4, 7, 6]
targets = [3, 4, 4, 3, 5, 6, 8, 7, 8, 9, 9]
values = [2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 2]


def nodify(node_names):
    node_names = unique_list
    # uniqe name endings
    ends = sorted(list(set([e[-1] for e in node_names])))

    # intervals
    steps = 1/len(ends)

    # x-values for each unique name ending
    # for input as node position
    nodes_x = {}
    xVal = 0
    for e in ends:
        nodes_x[str(e)] = xVal
        xVal += steps

    # x and y values in list form
    x_values = [nodes_x[n[-1]] for n in node_names]
    y_values = [0.1]*len(x_values)

    return x_values, y_values

nodified = nodify(node_names=unique_list)

# plotly setup
fig = go.Figure(data=[go.Sankey(
      arrangement='snap',
      node = dict(
      pad = 15,
      thickness = 20,
      line = dict(color = "black", width = 0.5),
      label = unique_list,
      color = "blue",
     x=nodified[0],
     y=nodified[1]
    ),
    link = dict(
      source = sources,
      target = targets,
      value = values
  ))])

fig.show()

In [ ]:
df = df_merge
df1 = df.groupby(['Irrigation Practice' , 'Crop'])['fail_share'].count().reset_index()
df1.columns = ['source', 'target', 'value']

df2 = df.groupby(['Crop' , 'SPEI9'])['fail_share'].count().reset_index()
df2.columns = ['source', 'target', 'value']

df1['source'] = df1.source.map({'I': 'Irigation', 'N': 'Rainfed'})
df2['target'] = df2.target.map({-1: 'Non-Drought', 1: 'Drought'})
links = pd.concat([df1, df2], axis=0)
unique_source_target = list(pd.unique(links[['source', 'target']].values.ravel('K')))
mapping_dict = {k: v for v, k in enumerate(unique_source_target)}
links['source'] = links['source'].map(mapping_dict)
links['target'] = links['target'].map(mapping_dict)

links_dict = links.to_dict(orient='list')
import plotly.graph_objects as go
import pandas as pd
color_palette0 = ['#8080ff','#ff6666']
color_palette1 = ['rgba(0,0,255, 0.35)'] * 8
color_palette2 = ['rgba(255,0,0, 0.35)'] * 8
color_palette3 = ['#737373'] * 8
color_palette4 = ['rgba(0,0,255, 0.35)','rgba(255,0,0, 0.35)'] * 8
color_palette5 = ['#8080ff','#ff6666']


fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=unique_source_target,
        color = color_palette0 + color_palette3 + color_palette5,
    ),
    link=dict(
        source=links_dict["source"],
        target=links_dict["target"],
        value=links_dict["value"],
        color = color_palette1 + color_palette2 + color_palette4
    )
)])

fig.update_layout(
    title_text="Failure Share",
    # font_family="Courier New",
    font_color="black",
    font_size=12,
    # title_font_family="Times New Roman",
    # title_font_color="blue",
    title_font_size=18,
)
fig.show()


# **Parcats Diagramm**

In [6]:
bins = [0, 0.25, 0.5, 0.75, 1]
labels = [1, 2, 3, 4]
df_svi = df_svi.drop(columns=['State','County']).set_index(['FIPS','year'])
df_svi_cat = df_svi.apply(lambda x: pd.cut(x, bins=bins, labels=labels), axis=0)
df_svi_cat = df_svi_cat.reset_index()


In [7]:
df_svi_cat = df_svi_cat[['FIPS','year','RPL_THEMES']]
cond_ig = df_fail['Irrigation Practice'] != 'ALL'
cond_fail =  df_fail.fail_share > 0
cols = ['FIPS', 'Crop', 'Irrigation Practice', 'year', 'fail_share']
df_merge = pd.merge(df_fail[cond_ig & cond_fail][cols],df_svi_cat,on=['FIPS','year'])
df_merge

,FIPS,Crop,Irrigation Practice,year,fail_share,RPL_THEMES
0,01001,COTTON,N,2019,0.006778,2
1,01001,COTTON,N,2017,0.005038,2
2,01001,SOYBEANS,N,2017,0.003043,2
3,01001,CORN,N,2017,0.035594,2
4,01003,COTTON,N,2017,0.002362,2
...,...,...,...,...,...,...
21814,56033,WHEAT,N,2013,0.021243,1
21815,56045,HAY,N,2017,0.024384,2
21816,56045,WHEAT,N,2021,0.021856,1
21817,56045,OATS,N,2011,0.047581,1


In [8]:
fpath_counties_clim = '<DATA_ROOT>/US_ClimateRegions/Counties_ClimReg.csv'
df_counties_clim = pd.read_csv(fpath_counties_clim)
df_counties_clim = df_counties_clim.drop(columns=['Unnamed: 0',	'State', 	'County'])
formatted_numbers = [str(n).zfill(5) for n in df_counties_clim['FIPS'].values]
df_counties_clim['FIPS'] = formatted_numbers
df_counties_clim

,FIPS,ClimRegNam,ClimRegCod
0,01001,Southeast,104.0
1,01003,Southeast,104.0
2,01005,Southeast,104.0
3,01007,Southeast,104.0
4,01009,Southeast,104.0
...,...,...,...
3103,56037,Northern Rockies and Plains,105.0
3104,56039,Northern Rockies and Plains,105.0
3105,56041,Northern Rockies and Plains,105.0
3106,56043,Northern Rockies and Plains,105.0


**Parcats Based on Crops (Climate regions aggregated)**

In [9]:
df_merge_clim = pd.merge(df_merge,df_counties_clim,on=['FIPS'])
grp_cols = ['ClimRegNam','Crop','Irrigation Practice','RPL_THEMES']
df_group = df_merge_clim.groupby(grp_cols).count().reset_index()
df_group = df_group[['ClimRegNam', 'Crop', 'Irrigation Practice', 'RPL_THEMES', 'FIPS']]
# df_group['ir_svi'] = df_group['Irrigation Practice'] + df_group['RPL_THEMES'].astype(str)
df_group = df_group.rename(columns={'FIPS':'fail_count'})
df_group

,ClimRegNam,Crop,Irrigation Practice,RPL_THEMES,fail_count
0,Northeast,BARLEY,I,1,0
1,Northeast,BARLEY,I,2,0
2,Northeast,BARLEY,I,3,0
3,Northeast,BARLEY,I,4,0
4,Northeast,BARLEY,N,1,14
...,...,...,...,...,...
571,West,WHEAT,I,4,45
572,West,WHEAT,N,1,0
573,West,WHEAT,N,2,0
574,West,WHEAT,N,3,6


In [10]:
df_merge_clim.Crop = df_merge_clim.Crop.map({
    'WHEAT': 0, 'CORN': 1,  'SOYBEANS': 2,   'COTTON': 3,
    'SORGHUM': 4,   'HAY': 5,   'OATS': 6,  'BARLEY': 7
})
# df_merge_clim#.ClimRegNam.unique()

In [ ]:
!pip install -U kaleido

In [13]:
import plotly.io as pio
import plotly.graph_objects as go

df_merge_clim = df_merge_clim.dropna()
crop_dim = go.parcats.Dimension(values = df_merge_clim.Crop , label="Crop",
                                categoryarray=[0,1,2,3,4,5,6,7],
                                ticktext=[
                                    'WHEAT','CORN','SOYBEANS','COTTON',
                                    'SORGHUM','HAY','OATS','BARLEY']
                                )
irig_dim = go.parcats.Dimension(values = df_merge_clim['Irrigation Practice'],
                                label="Irrigated", categoryarray=['I', 'N'],
                                ticktext=['Irrigated', 'Rainfed'])

svi_dim  = go.parcats.Dimension(values = df_merge_clim['RPL_THEMES'], label="SVI",
                                categoryarray=[1,2,3,4],
                                ticktext=['Low', 'Low-Medium', 'Medium-High', 'High'])

colors_crop = {
    'WHEAT': '#add8e6',       # Light Blue for young wheat, turning to '#f5deb3' for mature wheat
    'CORN': '#008000',        # Dark Green for corn
    'SOYBEANS': '#00ff00',    # Bright Green for soybeans
    'COTTON': '#bdbd37',      # Light Yellow for cotton
    'SORGHUM': '#ff4500',     # Reddish-Orange for sorghum
    'HAY': '#785705',         # Dark Goldenrod for hay
    'OATS': '#ccc9c4',        # Bisque for oats
    'BARLEY': '#f56c1d'       # Sandy Brown for barley
}

color_column = df_merge_clim.Crop
# colorscale = [[i, value] for i, value in enumerate(colors_crop.values())]
# colorscale = [[key, value] for key, value in colors_crop.items()]
# colorscale = list[colors_crop.values()]
colorscale = ['#377eb8','#4daf4a','#984ea3','#ff7f00','#ffff33','#a65628','#f781bf','#e41a1c',]
fig = go.Figure(data = [go.Parcats(dimensions=[crop_dim, irig_dim, svi_dim],
        line={'color': color_column, 'colorscale': colorscale},
        hoveron='color', hoverinfo='count+probability',
        labelfont={'size': 18, 'family': 'Times'},
        tickfont={'size': 16, 'family': 'Times'},
        arrangement='freeform')])
fig.update_layout(
    width=1200,  # Set the width of the figure in pixels
    height=800,  # Set the height of the figure in pixels
)

fig.show()
# out_path = '<DATA_ROOT>/Final_Exports/20231123-Section4/'
# # plt.savefig(out_path + 'Parcats_Diagramm_Crop_Irig_SVI' + '.png' , dpi=1200)
# pio.write_image(fig, out_path + 'Parcats_Diagramm_Crop_Irig_SVI' + '.png' , width=1200, height=800, scale=3)

**Parcats Based on Climate regions (Crops aggregated)**

In [ ]:
df_merge_clim = pd.merge(df_merge,df_counties_clim,on=['FIPS'])
grp_cols = ['ClimRegNam','Crop','Irrigation Practice','RPL_THEMES']
df_group = df_merge_clim.groupby(grp_cols).count().reset_index()
df_group = df_group[['ClimRegNam', 'Crop', 'Irrigation Practice', 'RPL_THEMES', 'FIPS']]
# df_group['ir_svi'] = df_group['Irrigation Practice'] + df_group['RPL_THEMES'].astype(str)
df_group = df_group.rename(columns={'FIPS':'fail_count'})
df_group

In [ ]:
df_merge_clim.ClimRegNam = df_merge_clim.ClimRegNam.map({
    'Southeast': 0, 'Southwest': 1,  'South': 2,   'West': 3,
    'Northeast': 4,   'Northwest': 5,   'Ohio Valley': 6,  'Upper Midwest': 7,
    'Northern Rockies and Plains' : 8
})
df_merge_clim = df_merge_clim.dropna()

In [ ]:
!pip install -U kaleido

In [ ]:
import plotly.io as pio
import plotly.graph_objects as go

# df_merge_clim = df_merge_clim.dropna()
clim_dim = go.parcats.Dimension(values = df_merge_clim.ClimRegNam , label="Climate Region",
                                categoryarray=[0,1,2,3,4,5,6,7,8],
                                ticktext=[
                                    'SE', 'SW', 'S', 'W', 'NE',
                                    'NW', 'C', 'ENC','WNC']
                                )
irig_dim = go.parcats.Dimension(values = df_merge_clim['Irrigation Practice'],
                                label="Irrigated", categoryarray=['I', 'N'],
                                ticktext=['Irrigated', 'Rainfed'])

svi_dim  = go.parcats.Dimension(values = df_merge_clim['RPL_THEMES'], label="SVI",
                                categoryarray=[1,2,3,4],
                                ticktext=['Low', 'Low-Medium', 'Medium-High', 'High'])

color_column = df_merge_clim.ClimRegNam
colorscale = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00','#ffff33','#a65628','#f781bf','#999999']


fig = go.Figure(data = [go.Parcats(dimensions=[clim_dim, irig_dim, svi_dim],
        line={'color': color_column, 'colorscale': colorscale},
        hoveron='color', hoverinfo='count+probability',
        labelfont={'size': 18, 'family': 'Times'},
        tickfont={'size': 16, 'family': 'Times'},
        arrangement='freeform')])
fig.update_layout(
    width=1200,  # Set the width of the figure in pixels
    height=800,  # Set the height of the figure in pixels
)


fig.show()
# out_path = '<DATA_ROOT>/Final_Exports/20231123-Section4/'
# # plt.savefig(out_path + 'Parcats_Diagramm_Climate_Irig_SVI' + '.png' , dpi=1200)
# pio.write_image(fig, out_path + 'Parcats_Diagramm_Climate_Irig_SVI' + '.png' , width=1200, height=800, scale=3)

# **Pie Chart - Filure & Climate Region**

In [ ]:
fsvi = '<DATA_ROOT>/SVI/SVI_timeseries.csv'
df_svi = pd.read_csv(fsvi)
df_svi = df_svi.sort_values(by=['FIPS'])
formatted_numbers = [str(n).zfill(5) for n in df_svi['FIPS'].values]
df_svi['FIPS'] = formatted_numbers
df_svi.iloc[0:5,:]

In [ ]:
cols = ['FIPS','RPL_THEMES']
df_svi_avg = df_svi[cols]
df_svi_avg = df_svi_avg.groupby('FIPS').mean()
df_svi_avg

In [ ]:
bins = [0, 0.25, 0.5, 0.75, 1]
labels = [1, 2, 3, 4]
# df_svi = df_svi.drop(columns=['State','County']).set_index(['FIPS','year'])
df_svi_cat = df_svi_avg.apply(lambda x: pd.cut(x, bins=bins, labels=labels), axis=0)
df_svi_cat = df_svi_cat.reset_index()
df_svi_cat

In [ ]:
fpath_counties_clim = '<DATA_ROOT>/US_ClimateRegions/Counties_ClimReg.csv'
df_counties_clim = pd.read_csv(fpath_counties_clim)
df_counties_clim = df_counties_clim.drop(columns=['Unnamed: 0',	'State', 	'County'])
formatted_numbers = [str(n).zfill(5) for n in df_counties_clim['FIPS'].values]
df_counties_clim['FIPS'] = formatted_numbers
df_counties_clim

In [ ]:
df_cnt_svi = pd.merge(df_counties_clim , df_svi_cat,on = ['FIPS'])
df_group = df_cnt_svi.groupby(['ClimRegNam','RPL_THEMES']).count().reset_index()
df_group

In [ ]:
dict_clim = {
    'Northeast':'NE', 'Northern Rockies and Plains':'WNC',
    'Northwest':'NW', 'Ohio Valley':'C' , 'South':'S' ,
    'Southeast':'SE', 'Southwest':'SW' , 'Upper Midwest':'ENC' , 'West':'W'
    }
df_group['ClimRegNam'] = df_group['ClimRegNam'].replace(dict_clim)
df_pivot = df_group.pivot(index='RPL_THEMES',columns='ClimRegNam',values='FIPS')
df_pivot

In [126]:
import matplotlib.patches as patches

colors =['#F4A460','#D2691E','#8B4513','#800000']
clims = list(df_pivot.columns) # dict_clim.keys()

fig , axes = plt.subplots(3,3,dpi=1200)
fig.subplots_adjust(hspace=0.9,wspace=-0.4)

for i , ax in enumerate(list(axes.flatten())):
  clim = clims[i]
  ax.set_title(clim, fontdict={'family': 'sans-serif',  'size': 7},y=1.15)
  ax.set_xlabel(str(df_pivot[clim].sum()), fontdict={'family': 'sans-serif',  'size': 7},labelpad=15)
  ax.pie(df_pivot[clim], startangle=90, colors=colors, radius=1.8)

out_path = '<DATA_ROOT>/Final_Exports/20231123-Section4/'
plt.savefig(out_path + 'PieChart_ClimReg_Counties' + '.png',dpi=1200)
plt.close()


In [ ]:
ffail = '<DATA_ROOT>/crop_failure_data/crops_failure_0595.csv'
df_fail = pd.read_csv(ffail)
df_fail = df_fail.drop(columns=['Unnamed: 0'])
df_fail = df_fail.sort_values(by=['FIPS'])
# cond1 = df_fail['year']>2011
cond2 = df_fail['Planted Acres'] > 0
df_fail = df_fail[cond2]
df_fail['fail_share'] = df_fail['Failed Acres'] /  df_fail['Planted Acres']
df_fail = df_fail[df_fail.fail_share < 1]
# df_fail = df_fail[df_fail.fail_share > 0]
cond = df_fail['Irrigation Practice'] == 'ALL'
df_fail = df_fail[cond]
formatted_numbers = [str(n).zfill(5) for n in df_fail['FIPS'].values]
df_fail['FIPS'] = formatted_numbers
df_fail

In [128]:
fsvi = '<DATA_ROOT>/SVI/SVI_timeseries.csv'
df_svi = pd.read_csv(fsvi)
df_svi = df_svi.sort_values(by=['FIPS'])
formatted_numbers = [str(n).zfill(5) for n in df_svi['FIPS'].values]
df_svi['FIPS'] = formatted_numbers


In [ ]:
bins = [0, 0.25, 0.5, 0.75, 1]
labels = [1, 2, 3, 4]
df_svi = df_svi.drop(columns=['State','County']).set_index(['FIPS','year'])
df_svi_cat = df_svi.apply(lambda x: pd.cut(x, bins=bins, labels=labels), axis=0)
df_svi_cat = df_svi_cat.reset_index()
df_svi_cat

In [ ]:
df_merge = pd.merge(df_fail,df_svi_cat[['FIPS',	'year', 'RPL_THEMES']],on=['FIPS',	'year'])
df_merge = df_merge[['Crop', 'Planted Acres','year','RPL_THEMES']]
grp_cols = ['Crop','year','RPL_THEMES']
df_group = df_merge.groupby(grp_cols).sum().reset_index()
grp_cols = ['Crop', 'RPL_THEMES']
df_group = df_group.groupby(grp_cols).mean().reset_index()
df_group[['Crop', 'Planted Acres','RPL_THEMES']]

In [131]:
df_pivot = df_group.pivot(index='RPL_THEMES',columns='Crop',values='Planted Acres')
df_pivot

Crop,BARLEY,CORN,COTTON,HAY,OATS,SORGHUM,SOYBEANS,WHEAT
RPL_THEMES,,,,,,,,
1,403153.189013,2.622281e+07,1.101701e+05,2.694168e+06,459169.844067,598433.080073,2.456267e+07,7.888925e+06
2,230433.772340,1.710395e+07,6.107165e+05,1.703274e+06,302916.696447,432553.886180,1.663867e+07,5.911100e+06
3,174041.018167,1.121215e+07,1.123024e+06,1.047808e+06,217958.169873,451311.840400,1.126631e+07,4.558696e+06
4,148990.588000,6.836003e+06,3.775414e+06,7.283141e+05,203627.474467,694468.720100,6.772670e+06,4.505976e+06


In [132]:
import matplotlib.patches as patches

colors =['#F4A460','#D2691E','#8B4513','#800000']
crops = list(df_pivot.columns) # dict_clim.keys()

fig , axes = plt.subplots(3,3,dpi=1200)
fig.subplots_adjust(hspace=0.9,wspace=-0.4)

for i , ax in enumerate(list(axes.flatten())):
  if i == 8:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.zorder = -1
    break
  crop = crops[i]
  ax.set_title(crop, fontdict={'family': 'sans-serif',  'size': 7},y=1.15)
  ax.set_xlabel(str(df_pivot[crop].sum()) , fontdict={'family': 'sans-serif',  'size': 7},labelpad=15)
  ax.pie(df_pivot[crop], startangle=90, colors=colors, radius=1.8)


out_path = '<DATA_ROOT>/Final_Exports/20231123-Section4/'
plt.savefig(out_path + 'PieChart_Crop_PlantedAnnual' + '.png',dpi=1200)
plt.close()
